In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. Deterministic Initialization
np.random.seed(42)
num_employees = 10000

# 2. Structural Arrays
employee_ids = [f"EMP-{10000 + i}" for i in range(num_employees)]
departments = ['Sales', 'Engineering', 'HR', 'Marketing', 'Finance']
salary_bands = ['Entry Level', 'Mid Level', 'Senior', 'Executive']
remote_freq = ['Never', 'Hybrid', 'Fully Remote']

data = []

# 3. Data Generation with Embedded Algorithmic Attrition Logic
for i in range(num_employees):
    dept = np.random.choice(departments, p=[0.3, 0.4, 0.05, 0.15, 0.1])
    salary = np.random.choice(salary_bands, p=[0.4, 0.4, 0.15, 0.05])
    remote = np.random.choice(remote_freq, p=[0.2, 0.6, 0.2])

    # Adding Hire Year to support Power BI Ribbon and Waterfall charts
    hire_year = np.random.randint(2010, 2024)
    tenure_years = 2024 - hire_year

    # Inject ~5% NULLs in Manager Satisfaction for SQL/Pandas cleaning practice
    mgr_sat = np.random.randint(1, 6) if np.random.rand() > 0.05 else np.nan

    promotions = np.random.choice([0, 1, 2], p=[0.6, 0.3, 0.1])
    overtime_hours = int(np.random.exponential(scale=15))

    # Base Salary Mapping for Financial Cost (ECOT)
    base_sal_map = {'Entry Level': 50000, 'Mid Level': 85000, 'Senior': 130000, 'Executive': 200000}
    actual_salary = base_sal_map[salary] + np.random.randint(-5000, 5000)

    # Mathematical Logic: Who quits?
    attrition_prob = 0.05
    if overtime_hours > 30: attrition_prob += 0.25
    if mgr_sat in [1, 2]: attrition_prob += 0.30
    if promotions == 0 and tenure_years > 3: attrition_prob += 0.20
    if remote == 'Never': attrition_prob += 0.10
    if salary == 'Entry Level': attrition_prob += 0.10

    attrition_prob = max(0.01, min(0.95, attrition_prob))
    attrition = 'Yes' if np.random.rand() < attrition_prob else 'No'

    data.append([
        employee_ids[i], dept, hire_year, salary, actual_salary, remote, tenure_years,
        mgr_sat, promotions, overtime_hours, attrition
    ])

# 4. Compile and Export Raw Data
columns = ['Employee_ID', 'Department', 'Hire_Year', 'Salary_Band', 'Annual_Salary', 'Remote_Work',
           'Tenure_Years', 'Manager_Satisfaction', 'Promotions_Last_5_Years', 'Overtime_Hours_Per_Month', 'Attrition']
df = pd.DataFrame(data, columns=columns)
df.to_csv('1_raw_hr_data.csv', index=False)

# 5. EXPLORATORY DATA ANALYSIS (EDA) AUTOMATION
sns.set_theme(style="whitegrid")

# EDA 1: Department-wise Attrition
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Department', hue='Attrition', palette={'No':'#2ecc71', 'Yes':'#e74c3c'})
plt.title('EDA: Department-wise Attrition Volume')
plt.tight_layout()
plt.savefig('EDA_1_Department.png', dpi=300)
plt.close()

# EDA 2: Salary Bands vs Attrition
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Salary_Band', hue='Attrition', palette={'No':'#3498db', 'Yes':'#e74c3c'}, order=salary_bands)
plt.title('EDA: Salary Band Attrition Vulnerability')
plt.tight_layout()
plt.savefig('EDA_2_Salary.png', dpi=300)
plt.close()

# EDA 3: Promotions vs Attrition
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Promotions_Last_5_Years', hue='Attrition', palette={'No':'#9b59b6', 'Yes':'#e74c3c'})
plt.title('EDA: Lack of Promotions Driving Turnover')
plt.tight_layout()
plt.savefig('EDA_3_Promotions.png', dpi=300)
plt.close()

print("SUCCESS: Data generated. 3 EDA visuals saved. Download from the left panel.")

SUCCESS: Data generated. 3 EDA visuals saved. Download from the left panel.


In [4]:
!pip install shap

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import shap

# 1. Load Data & Clean
df = pd.read_csv('1_raw_hr_data.csv')
df['Manager_Satisfaction'].fillna(df['Manager_Satisfaction'].median(), inplace=True)

# Convert Target to Binary
df['Attrition_Binary'] = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)

# 2. Preprocessing for ML
le_dept = LabelEncoder()
le_sal = LabelEncoder()
le_rem = LabelEncoder()

df['Dept_Encoded'] = le_dept.fit_transform(df['Department'])
df['Sal_Encoded'] = le_sal.fit_transform(df['Salary_Band'])
df['Rem_Encoded'] = le_rem.fit_transform(df['Remote_Work'])

features = ['Dept_Encoded', 'Sal_Encoded', 'Annual_Salary', 'Rem_Encoded',
            'Tenure_Years', 'Manager_Satisfaction', 'Promotions_Last_5_Years', 'Overtime_Hours_Per_Month']
X = df[features]
y = df['Attrition_Binary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Model Training (Random Forest Classification Model)
model = RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42)
model.fit(X_train, y_train)

# Accuracy & Confusion Matrix
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy Achieved: {accuracy * 100:.2f}%")
df['Attrition_Probability'] = model.predict_proba(X)[:, 1]

plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Stayed (0)', 'Quit (1)'], yticklabels=['Stayed (0)', 'Quit (1)'])
plt.title(f'Model Confusion Matrix (Accuracy: {accuracy*100:.1f}%)', fontweight='bold')
plt.ylabel('Actual Behavior')
plt.xlabel('AI Predicted Behavior')
plt.tight_layout()
plt.savefig('ML_1_Confusion_Matrix.png', dpi=300)
plt.close()

# 4. SHAP Value Analysis
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values[:, :, 1], X_test, feature_names=features, show=False)
plt.title('SHAP Analysis: Main Causes of Resignation', fontweight='bold')
plt.tight_layout()
plt.savefig('ML_2_SHAP_Analysis.png', dpi=300, bbox_inches='tight')
plt.close()

# 5. Feature Engineering: ECOT & Flight Risk
def assign_flight_risk(prob):
    if prob > 0.65: return 'High Risk'
    elif prob > 0.35: return 'Medium Risk'
    else: return 'Low Risk'

df['Flight_Risk_Tier'] = df['Attrition_Probability'].apply(assign_flight_risk)
df['Estimated_Cost_of_Turnover'] = df['Annual_Salary'] * 0.20 # 20% replacement cost

# Export Final Data for Power BI
columns_to_export = ['Employee_ID', 'Department', 'Hire_Year', 'Salary_Band', 'Annual_Salary', 'Remote_Work',
                     'Tenure_Years', 'Manager_Satisfaction', 'Promotions_Last_5_Years',
                     'Overtime_Hours_Per_Month', 'Attrition', 'Attrition_Probability',
                     'Flight_Risk_Tier', 'Estimated_Cost_of_Turnover']
df[columns_to_export].to_csv('2_final_hr_data_for_powerbi.csv', index=False)
print("SUCCESS: ML and SHAP Complete. Files saved.")

Model Accuracy Achieved: 71.90%
SUCCESS: ML and SHAP Complete. Files saved.
